## Task 1: Dataset Selection


## Import Libraries

In [3]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
import evaluate
import warnings
warnings.filterwarnings('ignore')

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Using device:", device)

PyTorch version: 2.11.0+cpu
CUDA available: False
Using device: cpu


## Task 2: Data Preprocessing

### 2.1 Load the CoNLL-2003 Dataset

In [4]:
# Load CoNLL-2003 dataset
# Uses the Parquet branch of eriktks/conll2003 — no loading script, works with latest datasets
from datasets import load_dataset

try:
    # Primary: eriktks mirror on the parquet branch (confirmed working)
    dataset = load_dataset("eriktks/conll2003", revision="convert/parquet")
except Exception:
    # Fallback: BramVanroy's clean republish (no loading script at all)
    dataset = load_dataset("BramVanroy/conll2003")

print(dataset)
print("\nFeatures:", dataset['train'].features)
print("\nSample:", dataset['train'][0])

conll2003/train/0000.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/312k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/283k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

Features: {'id': Value('string'), 'tokens': List(Value('string')), 'pos_tags': List(ClassLabel(names=['"', "''", '#', '$', '(', ')', ',', '.', ':', '``', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'LS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'NN|SYM', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB'])), 'chunk_tags': List(ClassLabel(names=['O', 'B-ADJP', 'I-ADJP', 'B-ADVP', 'I-ADVP', 'B-CONJP', 'I-CONJP', 'B-INTJ', 'I-INTJ', 'B-LST', 'I-LST', 'B-NP', 'I-NP', 'B-PP', 'I-PP', 'B-PRT', 'I-PRT', 'B-SB

In [5]:
# Extract label lists for POS and Chunking
pos_feature   = dataset['train'].features['pos_tags']
chunk_feature = dataset['train'].features['chunk_tags']

pos_label_list   = pos_feature.feature.names
chunk_label_list = chunk_feature.feature.names

print("POS Tags:", pos_label_list)
print(f"\nTotal POS labels: {len(pos_label_list)}")
print("\nChunk Tags:", chunk_label_list)
print(f"\nTotal Chunk labels: {len(chunk_label_list)}")

POS Tags: ['"', "''", '#', '$', '(', ')', ',', '.', ':', '``', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'LS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'NN|SYM', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB']

Total POS labels: 47

Chunk Tags: ['O', 'B-ADJP', 'I-ADJP', 'B-ADVP', 'I-ADVP', 'B-CONJP', 'I-CONJP', 'B-INTJ', 'I-INTJ', 'B-LST', 'I-LST', 'B-NP', 'I-NP', 'B-PP', 'I-PP', 'B-PRT', 'I-PRT', 'B-SBAR', 'I-SBAR', 'B-UCP', 'I-UCP', 'B-VP', 'I-VP']

Total Chunk labels: 23


### 2.2 Initialize Tokenizer

In [6]:
MODEL_CHECKPOINT = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
print("Tokenizer loaded:", MODEL_CHECKPOINT)

Tokenizer loaded: distilbert-base-uncased


### 2.3 Label Alignment – Handling Subwords and Special Tokens

BERT uses WordPiece tokenization, which can split a single word into multiple subword tokens.
For example: `"playing"` → `["play", "##ing"]`

**Strategy:**
- Assign the real label only to the **first subword** of each word
- Assign **-100** to all subsequent subwords and special tokens (`[CLS]`, `[SEP]`)
- `-100` is ignored by PyTorch's CrossEntropyLoss automatically

In [7]:
def tokenize_and_align_labels(examples, label_column):
    """
    Tokenizes input words using the BERT tokenizer and aligns
    the word-level labels to the subword-level tokens.

    Special tokens ([CLS], [SEP]) get label = -100 (ignored in loss).
    Continuation subwords (##) also get label = -100.
    Only the FIRST subword of each word carries the real label.
    """
    # Tokenize: is_split_into_words=True because input is already word-tokenized
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    all_labels = []
    for i, label_ids in enumerate(examples[label_column]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_id = None
        aligned_labels = []

        for word_id in word_ids:
            if word_id is None:
                # Special tokens: [CLS], [SEP] → ignore in loss
                aligned_labels.append(-100)
            elif word_id != previous_word_id:
                # First subword of a new word → assign real label
                aligned_labels.append(label_ids[word_id])
            else:
                # Continuation subword (e.g., ##ing) → ignore in loss
                aligned_labels.append(-100)

            previous_word_id = word_id

        all_labels.append(aligned_labels)

    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs

print("Label alignment function defined.")

Label alignment function defined.


### 2.4 Visualize Label Alignment on a Sample Sentence

In [8]:
# Show alignment for one example
sample = dataset['train'][0]
tokens = sample['tokens']
pos_tags = sample['pos_tags']

print("Original words:", tokens)
print("Original POS IDs:", pos_tags)
print("Original POS Tags:", [pos_label_list[t] for t in pos_tags])

# Tokenize just this sentence
tok_output = tokenizer(tokens, is_split_into_words=True)
subwords   = tokenizer.convert_ids_to_tokens(tok_output['input_ids'])
word_ids   = tok_output.word_ids()

print("\n{:<20} {:<12} {}".format("Subword Token", "Word ID", "Aligned POS"))
print("-" * 45)
prev = None
for sw, wid in zip(subwords, word_ids):
    if wid is None:
        label_str = "-100 (special)"
    elif wid != prev:
        label_str = pos_label_list[pos_tags[wid]]
    else:
        label_str = "-100 (subword)"
    print(f"{sw:<20} {str(wid):<12} {label_str}")
    prev = wid

Original words: ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']
Original POS IDs: [22, 42, 16, 21, 35, 37, 16, 21, 7]
Original POS Tags: ['NNP', 'VBZ', 'JJ', 'NN', 'TO', 'VB', 'JJ', 'NN', '.']

Subword Token        Word ID      Aligned POS
---------------------------------------------
[CLS]                None         -100 (special)
eu                   0            NNP
rejects              1            VBZ
german               2            JJ
call                 3            NN
to                   4            TO
boycott              5            VB
british              6            JJ
lamb                 7            NN
.                    8            .
[SEP]                None         -100 (special)


### 2.5 Preprocess Full Dataset

We create **two tokenized datasets** — one for POS tagging and one for Chunking.

In [9]:
from functools import partial

# --- POS Tagging Dataset ---
tokenized_pos = dataset.map(
    partial(tokenize_and_align_labels, label_column="pos_tags"),
    batched=True,
    remove_columns=dataset["train"].column_names
)

# --- Chunking Dataset ---
tokenized_chunk = dataset.map(
    partial(tokenize_and_align_labels, label_column="chunk_tags"),
    batched=True,
    remove_columns=dataset["train"].column_names
)

print("POS Dataset:", tokenized_pos)
print("\nChunk Dataset:", tokenized_chunk)

# Verify structure
sample_tok = tokenized_pos['train'][0]
print("\nTokenized sample keys:", list(sample_tok.keys()))
print("input_ids length:", len(sample_tok['input_ids']))
print("attention_mask:", sample_tok['attention_mask'][:10])
print("labels:", sample_tok['labels'][:10])

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

POS Dataset: DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3453
    })
})

Chunk Dataset: DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3453
    })
})

Tokenized sample keys: ['input_ids', 'token_type_ids', 'attention_mask', 'labels']
input_ids length: 11
attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1

## Task 3: Model Setup

We use **DistilBERT** – a lighter, faster version of BERT with ~97% of its performance.

### 3.1 POS Tagging Model

In [10]:
# Build label mappings for POS Tagging
pos_id2label = {i: label for i, label in enumerate(pos_label_list)}
pos_label2id = {label: i for i, label in enumerate(pos_label_list)}

print(f"Number of POS labels: {len(pos_label_list)}")
print("POS id2label (first 5):", dict(list(pos_id2label.items())[:5]))

# Load DistilBERT for token classification (POS)
pos_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(pos_label_list),
    id2label=pos_id2label,
    label2id=pos_label2id
)
print("\nPOS model loaded. Output layer size:", pos_model.classifier.out_features)

Number of POS labels: 47
POS id2label (first 5): {0: '"', 1: "''", 2: '#', 3: '$', 4: '('}


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



POS model loaded. Output layer size: 47


### 3.2 Chunking Model

In [11]:
# Build label mappings for Chunking
chunk_id2label = {i: label for i, label in enumerate(chunk_label_list)}
chunk_label2id = {label: i for i, label in enumerate(chunk_label_list)}

print(f"Number of Chunk labels: {len(chunk_label_list)}")
print("Chunk id2label:", chunk_id2label)

# Load DistilBERT for token classification (Chunking)
chunk_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(chunk_label_list),
    id2label=chunk_id2label,
    label2id=chunk_label2id
)
print("\nChunk model loaded. Output layer size:", chunk_model.classifier.out_features)

Number of Chunk labels: 23
Chunk id2label: {0: 'O', 1: 'B-ADJP', 2: 'I-ADJP', 3: 'B-ADVP', 4: 'I-ADVP', 5: 'B-CONJP', 6: 'I-CONJP', 7: 'B-INTJ', 8: 'I-INTJ', 9: 'B-LST', 10: 'I-LST', 11: 'B-NP', 12: 'I-NP', 13: 'B-PP', 14: 'I-PP', 15: 'B-PRT', 16: 'I-PRT', 17: 'B-SBAR', 18: 'I-SBAR', 19: 'B-UCP', 20: 'I-UCP', 21: 'B-VP', 22: 'I-VP'}


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Chunk model loaded. Output layer size: 23


## Task 4: Training

We use Hugging Face `Trainer` with:
- Learning rate: `2e-5` (standard for BERT fine-tuning)
- Epochs: `3`
- Batch size: `16`
- Weight decay regularization

In [12]:
# Data collator pads sequences in a batch to the same length dynamically
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# Load seqeval for sequence-based evaluation
seqeval = evaluate.load("seqeval")

print("Data collator and seqeval metric loaded.")

Data collator and seqeval metric loaded.


In [13]:
def build_compute_metrics(label_list):
    """
    Factory function: returns a compute_metrics function specific to
    the given label_list (POS or Chunk).
    Uses seqeval which expects IOB-format lists of label strings.
    """
    def compute_metrics(p):
        predictions, labels = p
        # Get the predicted class for each token (argmax over vocab)
        predictions = np.argmax(predictions, axis=2)

        # Convert integer IDs → label strings, skipping -100
        true_predictions = [
            [label_list[pred] for pred, lab in zip(prediction, label)
             if lab != -100]
            for prediction, label in zip(predictions, labels)
        ]
        true_labels = [
            [label_list[lab] for lab in label if lab != -100]
            for label in labels
        ]

        results = seqeval.compute(
            predictions=true_predictions,
            references=true_labels
        )
        return {
            "precision": results["overall_precision"],
            "recall":    results["overall_recall"],
            "f1":        results["overall_f1"],
            "accuracy":  results["overall_accuracy"]
        }
    return compute_metrics

print("Metrics function factory defined.")

Metrics function factory defined.


### 4.1 Train POS Tagging Model

In [14]:
pos_args = TrainingArguments(
    output_dir="./pos_model",
    eval_strategy="epoch",           # Renamed from evaluation_strategy in newer versions
    save_strategy="epoch",
    learning_rate=2e-5,              # Standard BERT fine-tuning LR
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,               # L2 regularization
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=200,
    report_to="none"                 # Disable wandb logging
)

pos_trainer = Trainer(
    model=pos_model,
    args=pos_args,
    train_dataset=tokenized_pos["train"],
    eval_dataset=tokenized_pos["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=build_compute_metrics(pos_label_list)
)

print("Starting POS model training...")
pos_trainer.train()
print("POS training complete!")

Starting POS model training...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.253655,0.258806,0.909062,0.910275,0.909668,0.937347
2,0.196682,0.229695,0.919410,0.916220,0.917812,0.943246
3,0.160144,0.222571,0.920858,0.918646,0.919751,0.944258


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


POS training complete!


### 4.2 Train Chunking Model

In [15]:
chunk_args = TrainingArguments(
    output_dir="./chunk_model",
    eval_strategy="epoch",           # Renamed from evaluation_strategy in newer versions
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=200,
    report_to="none"
)

chunk_trainer = Trainer(
    model=chunk_model,
    args=chunk_args,
    train_dataset=tokenized_chunk["train"],
    eval_dataset=tokenized_chunk["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=build_compute_metrics(chunk_label_list)
)

print("Starting Chunking model training...")
chunk_trainer.train()
print("Chunking training complete!")

Starting Chunking model training...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.212047,0.200164,0.904041,0.902320,0.903179,0.949321
2,0.154472,0.179588,0.915136,0.909624,0.912372,0.953701
3,0.132952,0.172957,0.916104,0.911761,0.913927,0.954850


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Chunking training complete!


## Task 5: Evaluation

We evaluate both models on the **test set** using `seqeval`, which computes:
- **Precision** – Of all predicted labels, how many are correct?
- **Recall** – Of all true labels, how many did we find?
- **F1 Score** – Harmonic mean of precision and recall

In [21]:
# --- POS Evaluation on Test Set ---
print("=" * 50)
print("POS TAGGING EVALUATION (Test Set)")
print("=" * 50)

# Use predict() instead of evaluate() — no callback state required
pos_preds_output = pos_trainer.predict(tokenized_pos["test"])
pos_results = build_compute_metrics(pos_label_list)(
    (pos_preds_output.predictions, pos_preds_output.label_ids)
)
pos_results = {f"eval_{k}": v for k, v in pos_results.items()}
for k, v in pos_results.items():
    print(f"  {k:30s}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

POS TAGGING EVALUATION (Test Set)
  eval_precision                : 0.9120
  eval_recall                   : 0.9073
  eval_f1                       : 0.9097
  eval_accuracy                 : 0.9388


In [22]:
# --- Chunking Evaluation on Test Set ---
print("=" * 50)
print("CHUNKING EVALUATION (Test Set)")
print("=" * 50)

chunk_preds_output = chunk_trainer.predict(tokenized_chunk["test"])
chunk_results = build_compute_metrics(chunk_label_list)(
    (chunk_preds_output.predictions, chunk_preds_output.label_ids)
)
chunk_results = {f"eval_{k}": v for k, v in chunk_results.items()}
for k, v in chunk_results.items():
    print(f"  {k:30s}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

CHUNKING EVALUATION (Test Set)


  eval_precision                : 0.9071
  eval_recall                   : 0.8971
  eval_f1                       : 0.9021
  eval_accuracy                 : 0.9513


In [ ]:
# Summary comparison table
print("\n" + "=" * 55)
print(f"{'Metric':<20} {'POS Tagging':>15} {'Chunking':>15}")
print("-" * 55)
for metric in ['eval_precision', 'eval_recall', 'eval_f1', 'eval_accuracy']:
    m_short = metric.replace('eval_', '').capitalize()
    p_val = pos_results.get(metric, 'N/A')
    c_val = chunk_results.get(metric, 'N/A')
    p_str = f"{p_val:.4f}" if isinstance(p_val, float) else str(p_val)
    c_str = f"{c_val:.4f}" if isinstance(c_val, float) else str(c_val)
    print(f"{m_short:<20} {p_str:>15} {c_str:>15}")
print("=" * 55)

## Task 6: Inference

### 6.1 Inference Helper Function

In [23]:
def predict_tags(sentence, model, tokenizer, id2label):
    """
    Predict token-level tags for a raw input sentence.

    Steps:
    1. Split sentence into words
    2. Tokenize with BERT tokenizer (subwords)
    3. Run forward pass through model
    4. Decode predictions → label strings
    5. Assign only the first subword prediction to each word
    """
    words = sentence.split()

    # Tokenize
    inputs = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True
    )
    word_ids = inputs.word_ids()

    # Move to correct device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    model.to(device)

    # Forward pass (no gradient needed during inference)
    with torch.no_grad():
        outputs = model(**inputs)

    # Decode predictions
    predictions = torch.argmax(outputs.logits, dim=-1).squeeze().tolist()

    # Map: for each original word, take the prediction from its first subword
    word_predictions = {}
    for token_idx, word_id in enumerate(word_ids):
        if word_id is not None and word_id not in word_predictions:
            word_predictions[word_id] = id2label[predictions[token_idx]]

    return words, [word_predictions[i] for i in range(len(words))]

print("Inference function ready.")

Inference function ready.


### 6.2 Run Inference on Custom Sentences

In [24]:
# Test sentences
test_sentences = [
    "John works at Google in California",
    "The quick brown fox jumps over the lazy dog",
    "She is studying machine learning at the university"
]

for sent in test_sentences:
    print("\n" + "=" * 60)
    print(f"Input: {sent}")
    print("-" * 60)

    # POS Prediction
    words, pos_preds = predict_tags(
        sent, pos_model, tokenizer, pos_id2label
    )
    print("POS Tags:")
    print(f"  {' '.join(f'{w}/{t}' for w, t in zip(words, pos_preds))}")

    # Chunk Prediction
    _, chunk_preds = predict_tags(
        sent, chunk_model, tokenizer, chunk_id2label
    )
    print("Chunk Tags:")
    print(f"  {' '.join(f'{w}/{t}' for w, t in zip(words, chunk_preds))}")


Input: John works at Google in California
------------------------------------------------------------
POS Tags:
  John/NNP works/VBZ at/IN Google/NNP in/IN California/NNP
Chunk Tags:
  John/B-NP works/B-VP at/B-PP Google/B-NP in/B-PP California/B-NP

Input: The quick brown fox jumps over the lazy dog
------------------------------------------------------------
POS Tags:
  The/DT quick/JJ brown/NNP fox/NNP jumps/VBZ over/IN the/DT lazy/JJ dog/NN
Chunk Tags:
  The/B-NP quick/I-NP brown/I-NP fox/I-NP jumps/B-VP over/B-PP the/B-NP lazy/I-NP dog/I-NP

Input: She is studying machine learning at the university
------------------------------------------------------------
POS Tags:
  She/PRP is/VBZ studying/VBG machine/NN learning/NN at/IN the/DT university/NN
Chunk Tags:
  She/B-NP is/B-VP studying/I-VP machine/B-NP learning/I-NP at/B-PP the/B-NP university/I-NP


### 6.3 Formatted Output for Sample Sentence

In [25]:
# Detailed table view for the main example
main_sent = "John works at Google in California"
words, pos_preds   = predict_tags(main_sent, pos_model, tokenizer, pos_id2label)
_, chunk_preds     = predict_tags(main_sent, chunk_model, tokenizer, chunk_id2label)

print(f"\nSentence: '{main_sent}'\n")
print(f"{'Word':<15} {'POS Tag':<12} {'Chunk Tag'}")
print("-" * 40)
for w, p, c in zip(words, pos_preds, chunk_preds):
    print(f"{w:<15} {p:<12} {c}")


Sentence: 'John works at Google in California'

Word            POS Tag      Chunk Tag
----------------------------------------
John            NNP          B-NP
works           VBZ          B-VP
at              IN           B-PP
Google          NNP          B-NP
in              IN           B-PP
California      NNP          B-NP


## Task 7: Comparison – POS Tagging vs Chunking

### 7.1 Conceptual Comparison

In [26]:
comparison = {
    "Aspect": [
        "Task Level",
        "Granularity",
        "Label Format",
        "Example Labels",
        "Difficulty",
        "Use Case",
        "Context Needed",
        "Model F1 (ours)"
    ],
    "POS Tagging": [
        "Grammar-level (word)",
        "Per-word syntactic role",
        "Single flat labels",
        "NN, VBZ, NNP, JJ, IN",
        "Easier – 45 labels",
        "Spell checking, parsing, IR",
        "Local context (1-2 tokens)",
        f"{pos_results.get('eval_f1', 'N/A'):.4f}" if isinstance(pos_results.get('eval_f1'), float) else "See eval"
    ],
    "Chunking": [
        "Phrase-level (multi-word)",
        "Multi-word phrase spans",
        "IOB2 (B-/I-/O)",
        "B-NP, I-NP, B-VP, O",
        "Medium – phrase spans",
        "Info extraction, Q&A, MT",
        "Phrase-span context",
        f"{chunk_results.get('eval_f1', 'N/A'):.4f}" if isinstance(chunk_results.get('eval_f1'), float) else "See eval"
    ]
}

print("\nPOS TAGGING vs CHUNKING – COMPARISON")
print("=" * 80)
print(f"{'Aspect':<22} {'POS Tagging':<30} {'Chunking'}")
print("-" * 80)
for a, p, c in zip(comparison["Aspect"], comparison["POS Tagging"], comparison["Chunking"]):
    print(f"{a:<22} {p:<30} {c}")
print("=" * 80)


POS TAGGING vs CHUNKING – COMPARISON
Aspect                 POS Tagging                    Chunking
--------------------------------------------------------------------------------
Task Level             Grammar-level (word)           Phrase-level (multi-word)
Granularity            Per-word syntactic role        Multi-word phrase spans
Label Format           Single flat labels             IOB2 (B-/I-/O)
Example Labels         NN, VBZ, NNP, JJ, IN           B-NP, I-NP, B-VP, O
Difficulty             Easier – 45 labels             Medium – phrase spans
Use Case               Spell checking, parsing, IR    Info extraction, Q&A, MT
Context Needed         Local context (1-2 tokens)     Phrase-span context
Model F1 (ours)        0.9097                         0.9021


In [27]:
# Show on same sentence
example = "The big company hired skilled engineers"
words, pos = predict_tags(example, pos_model, tokenizer, pos_id2label)
_, chunks  = predict_tags(example, chunk_model, tokenizer, chunk_id2label)

print(f"\nSentence: '{example}'")
print("\nPOS Tagging (word-level syntactic roles):")
for w, p in zip(words, pos):
    print(f"  {w:12} → {p}")

print("\nChunking (phrase-span grouping):")
for w, c in zip(words, chunks):
    print(f"  {w:12} → {c}")

print("\nObservation:")
print("  POS assigns individual grammatical roles per word.")
print("  Chunking groups words into phrases (NP='noun phrase', VP='verb phrase', etc.)")
print("  B- marks phrase start, I- marks continuation, O marks outside any phrase.")


Sentence: 'The big company hired skilled engineers'

POS Tagging (word-level syntactic roles):
  The          → DT
  big          → JJ
  company      → NN
  hired        → VBD
  skilled      → JJ
  engineers    → NNS

Chunking (phrase-span grouping):
  The          → B-NP
  big          → I-NP
  company      → I-NP
  hired        → B-VP
  skilled      → B-NP
  engineers    → I-NP

Observation:
  POS assigns individual grammatical roles per word.
  Chunking groups words into phrases (NP='noun phrase', VP='verb phrase', etc.)
  B- marks phrase start, I- marks continuation, O marks outside any phrase.


## Task 8: Report / Blog

Differences Between POS Tagging and Chunking
POS tagging assigns a grammatical label to each word in a sentence. For example, whether a word is a noun, verb, adjective, etc. It works at the word level and looks at individual tokens one at a time.
Chunking is a step above that. Instead of labeling single words, it groups words together into phrases like noun phrases (NP) or verb phrases (VP). It uses IOB2 format where B- means the start of a phrase, I- means continuation, and O means the word is outside any phrase.
So for a sentence like "The big company hired skilled engineers":

POS gives: The/DT, big/JJ, company/NN, hired/VBD, skilled/JJ, engineers/NNS
Chunking gives: [B-NP, I-NP, I-NP], [B-VP], [B-NP, I-NP]


Challenges Faced

Subword alignment was the trickiest part. DistilBERT splits words into pieces (e.g. "playing" → "play", "##ing") but we only want one label per original word. The fix was to label only the first subword and assign -100 to the rest, which PyTorch ignores in the loss.
Special tokens like [CLS] and [SEP] also get -100 for the same reason.
Label count difference — POS has 45 labels while chunking has only 23, so two separate model heads were needed.
seqeval expects string labels not integers, and also expects IOB format. POS tags don't follow IOB but seqeval still handles overall metrics fine.


Observations
BERT's bidirectional attention works really well for both tasks. POS tagging is easier because each word's label mostly depends on nearby words. Chunking is slightly harder since you need to understand phrase boundaries across multiple tokens, but BERT handles this well with self-attention.
Transfer learning made a big difference here. Getting strong F1 in just 3 epochs with lr=2e-5 shows how much the pre-trained weights already know.

Key Takeaways

Pre-trained BERT weights give a huge head start — fine-tuning is fast and effective
Label alignment with -100 is critical, wrong alignment silently breaks training
seqeval is better than accuracy for this task since accuracy counts padding tokens too
The same DistilBERT backbone works for both POS and chunking, only the output head changes

## Save Models

In [ ]:
# Save both fine-tuned models and tokenizer
pos_model.save_pretrained("./saved_pos_model")
chunk_model.save_pretrained("./saved_chunk_model")
tokenizer.save_pretrained("./saved_tokenizer")
print("Models and tokenizer saved successfully.")

## Load Saved Model and Run Inference

In [ ]:
# Demonstrate loading a saved model
loaded_pos_model = AutoModelForTokenClassification.from_pretrained("./saved_pos_model")
loaded_tokenizer = AutoTokenizer.from_pretrained("./saved_tokenizer")
loaded_id2label  = loaded_pos_model.config.id2label

sentence = "Barack Obama was the president of the United States"
words, preds = predict_tags(sentence, loaded_pos_model, loaded_tokenizer, loaded_id2label)

print(f"\nInference with loaded POS model:")
print(f"Input: {sentence}")
print("\n{:<15} {}".format("Word", "POS Tag"))
print("-" * 30)
for w, p in zip(words, preds):
    print(f"{w:<15} {p}")

---
## Summary

| Task | Status |
|------|--------|
| Dataset Selection (CoNLL-2003) | ✅ Complete |
| Data Preprocessing + Label Alignment | ✅ Complete |
| DistilBERT Model Setup (POS + Chunk) | ✅ Complete |
| Training with Hugging Face Trainer | ✅ Complete |
| Evaluation using seqeval (P/R/F1) | ✅ Complete |
| Inference on Custom Sentences | ✅ Complete |
| Comparison of POS vs Chunking | ✅ Complete |
| Report / Blog | ✅ Complete |

---
*Submitted by Om | Innomatics Research Labs – Agentic AI Intern*